In [20]:


import kagglehub
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers.tokenization_utils_base import BatchEncoding
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

In [21]:

path = kagglehub.dataset_download("pythonafroz/medquad-medical-question-answer-for-ai-research")
print("Path to dataset files:", path)

Path to dataset files: /home/codespace/.cache/kagglehub/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research/versions/1


In [22]:

df = pd.read_csv(f"{path}/medquad.csv")
df.head()

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [23]:

df_amostra = df.sample(5000)

In [24]:

nome_modelo = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(nome_modelo)
model = AutoModel.from_pretrained(nome_modelo)

In [25]:
def get_tokens(pergunta: str) -> BatchEncoding:
    return tokenizer(pergunta, return_tensors="pt")

In [26]:
def get_vetores(tokens_pergunta: BatchEncoding) -> Tensor:
    with torch.no_grad():
        outputs = model(**tokens_pergunta)
        embeddings = outputs.last_hidden_state
       
        mean_embedding = embeddings.mean(dim=1)
    return mean_embedding

In [27]:
df_amostra['tokens'] = df_amostra['question'].apply(lambda x: get_tokens(x))
df_amostra['vetores'] = df_amostra['tokens'].apply(lambda x: get_vetores(x))

In [28]:
def buscar_resposta(pergunta_usuario):
    tokens_usuario = get_tokens(pergunta_usuario)
    embedding_usuario = get_vetores(tokens_usuario)
    
    similaridades = []
    for i, embedding_dataset in enumerate(df_amostra['vetores']):
        similaridade = F.cosine_similarity(embedding_usuario, embedding_dataset, dim=1)
        similaridades.append((i, similaridade.item()))
    
    idx_mais_similar = max(similaridades, key=lambda x: x[1])[0]
    return df_amostra.iloc[idx_mais_similar]['answer']

#### Exemplo de perguntas que um usuario fazeria

In [29]:
pergunta_teste = "What are the symptoms of diabetes?"
resposta = buscar_resposta(pergunta_teste)
print(f"Pergunta: {pergunta_teste}")
print(f"Resposta: {resposta}")

Pergunta: What are the symptoms of diabetes?
Resposta: What are the symptoms of brittle diabetes? The main symptom of brittle diabetes is severe instability of blood glucose levels with frequent and unpredictable episodes of hypoglycemia and/or ketoacidosis that cause a disruption of daily activities. Three clinical presentations have been described: Predominant hyperglycemia with recurrent ketoacidosis, Predominant hypoglycemia, and Mixed hyper- and hypoglycemia. Patients with brittle diabetes have wide swings in their blood sugar levels and often experience differing blood sugar responses to the same dose and type of insulin. Complications such as neuropathy, nephropathy, and retinopathy are common. Most patients are females in their twenties of thirties, though any age or gender can be affected.


In [31]:
pergunta_teste = "What are the symptoms of leukemia?"
resposta = buscar_resposta(pergunta_teste)
print(f"Pergunta: {pergunta_teste}")
print(f"Resposta: {resposta}")

Pergunta: What are the symptoms of leukemia?
Resposta: Common symptoms of leukemia may include -  fevers  - frequent infections  - feeling weak or tired  -  headache  - bleeding and bruising easily  - pain in the bones or joints  -  swelling or discomfort in the abdomen (from an enlarged spleen)  -  swollen lymph nodes, especially in the neck or armpit  - weight loss.  fevers frequent infections feeling weak or tired headache bleeding and bruising easily pain in the bones or joints swelling or discomfort in the abdomen (from an enlarged spleen) swollen lymph nodes, especially in the neck or armpit weight loss. Symptoms of acute leukemia may include vomiting, confusion, loss of muscle control, and seizures.


In [ ]:
pergunta_teste = "What are the symptoms of pneumonia?"
resposta = buscar_resposta(pergunta_teste)
print(f"Pergunta: {pergunta_teste}")
print(f"Resposta: {resposta}")

Pergunta: What are the symptoms of cold?
Resposta: The signs and symptoms of pneumonia vary from mild to severe. Many factors affect how serious pneumonia is, including the type of germ causing the infection and your age and overall health. (For more information, go to "Who Is at Risk for Pneumonia?")
                
See your doctor promptly if you:
                
Have a high fever
                
Have shaking chills
                
Have a cough with phlegm (a slimy substance), which doesn't improve or worsens
                
Develop shortness of breath with normal daily activities
                
Have chest pain when you breathe or cough
                
Feel suddenly worse after a cold or the flu
                
People who have pneumonia may have other symptoms, including nausea (feeling sick to the stomach), vomiting, and diarrhea.
                
Symptoms may vary in certain populations. Newborns and infants may not show any signs of the infection. Or, they may vomit, have

In [34]:
pergunta_teste = "What are the symptoms of black plague?"
resposta = buscar_resposta(pergunta_teste)
print(f"Pergunta: {pergunta_teste}")
print(f"Resposta: {resposta}")

Pergunta: What are the symptoms of black plague?
Resposta: Common symptoms of leukemia may include -  fevers  - frequent infections  - feeling weak or tired  -  headache  - bleeding and bruising easily  - pain in the bones or joints  -  swelling or discomfort in the abdomen (from an enlarged spleen)  -  swollen lymph nodes, especially in the neck or armpit  - weight loss.  fevers frequent infections feeling weak or tired headache bleeding and bruising easily pain in the bones or joints swelling or discomfort in the abdomen (from an enlarged spleen) swollen lymph nodes, especially in the neck or armpit weight loss. Symptoms of acute leukemia may include vomiting, confusion, loss of muscle control, and seizures.
